In [1]:
import torch
import torchvision
import torchvision.transforms as transforms

transform = transforms.ToTensor()

train_dataset = torchvision.datasets.MNIST(
    root='./data', train=True, transform=transform, download=True
)

test_dataset = torchvision.datasets.MNIST(
    root='./data', train=False, transform=transform
)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64)

100%|██████████| 9.91M/9.91M [10:41<00:00, 15.4kB/s]
100%|██████████| 28.9k/28.9k [00:03<00:00, 7.29kB/s]
100%|██████████| 1.65M/1.65M [02:21<00:00, 11.6kB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 16.3kB/s]


In [ ]:
import torch.nn as nn
import torch.optim as optim

class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )

        self.fc = nn.Sequential(
            nn.Linear(64 * 14 * 14, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

In [3]:
model = CNN()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 5
for epoch  in range(epochs):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f'Epoch {epochs+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}')

Epoch 6/5, Loss: 0.2300
Epoch 6/5, Loss: 0.0865
Epoch 6/5, Loss: 0.0651
Epoch 6/5, Loss: 0.0518
Epoch 6/5, Loss: 0.0422


In [5]:
model.eval()

correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Test Accuracy: {correct/total:.4f}')

print(correct, total)

Test Accuracy: 0.9908
9908 10000


In [6]:
class RNN(nn.Module):
    def __init__(self):
        super(RNN, self).__init__()
        
        self.rnn = nn.RNN(28, 128, batch_first=True)
        self.fc = nn.Linear(128, 10)

    def forward(self, x):
        x = x.squeeze(1)  # (batch, 28, 28)
        
        out, _ = self.rnn(x)
        out = out[:, -1, :]
        
        out = self.fc(out)
        return out
    

model = RNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 5
for epoch in range(epochs):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f'Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}')

model.eval()

correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Test Accuracy: {correct/total:.4f}')

print(correct, total)

Epoch 1/5, Loss: 0.7130
Epoch 2/5, Loss: 0.3368
Epoch 3/5, Loss: 0.2577
Epoch 4/5, Loss: 0.2112
Epoch 5/5, Loss: 0.1841
Test Accuracy: 0.9570
9570 10000


In [7]:
from sklearn.metrics import confusion_matrix
import numpy as np

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        
        all_preds.extend(predicted.numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
print(cm)

[[ 967    0    1    0    0    3    6    1    2    0]
 [   0 1121    4    4    0    0    2    1    2    1]
 [   3    0  986   11    4    0    4   15    9    0]
 [   0    3    6  973    0   12    0    7    3    6]
 [   1    2    3    0  937    0   13    1    1   24]
 [   9    3    0   20    5  829    7    1    6   12]
 [   8    3    3    1    7    3  928    0    5    0]
 [   2   15   11    4    2    0    0  977    3   14]
 [   3    3    7    9   18    4    7    5  905   13]
 [   5    5    1   13   23    5    0    7    3  947]]


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

RNN(
  (rnn): RNN(28, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=10, bias=True)
)